# 03 Rule Classification

Apply deterministic first-pass rules to the latest inventory output and produce a review table. Still dry-run only.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

POLICY_PATH = PROJECT_ROOT / 'policy' / 'master_policy.yaml'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('POLICY_PATH =', POLICY_PATH)


PROJECT_ROOT = c:\00_dev\SCH-FILE-ORGANIZER
POLICY_PATH = c:\00_dev\SCH-FILE-ORGANIZER\policy\master_policy.yaml


In [2]:
from datetime import datetime
import pandas as pd

from src.policy_loader import PolicyLoader
from src.rules import classify_inventory, save_rule_outputs

policy = PolicyLoader.from_file(POLICY_PATH)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
inventory_files = list(OUTPUT_DIR.glob('inventory_*.parquet'))
assert inventory_files, 'No inventory parquet files found. Run 02_inventory.ipynb first.'

# Pick the newest file by filesystem timestamp, not filename order.
latest_inventory = max(inventory_files, key=lambda p: p.stat().st_mtime)
print('Using inventory:', latest_inventory.name)
inv = pd.read_parquet(latest_inventory)
print('Rows:', len(inv))
print('Columns:', list(inv.columns))


Using inventory: inventory_BCH01p800-01_vipe_SERRES_20260330_083028.parquet
Rows: 249
Columns: ['scan_root', 'absolute_path', 'relative_path', 'parent_relative', 'filename', 'stem', 'suffix', 'size_bytes', 'modified_at', 'created_at', 'depth_segments', 'path_length', 'filename_length', 'is_hidden', 'is_symlink', 'top_segment', 'hash', 'is_duplicate_hash', 'duplicate_group_size']


## Schema note
`classify_inventory()` now backfills missing inventory fields such as `filename`, `suffix`, `parent_relative`, `path_length`, and `filename_length` if you loaded an older inventory parquet. For best consistency, rerun `02_inventory.ipynb` after policy or scanner changes.


In [4]:
classified = classify_inventory(inv, POLICY_PATH)
classified[['relative_path', 'rule_status', 'rule_reason', 'rule_confidence', 'proposed_relative_target']].head(5)

,relative_path,rule_status,rule_reason,rule_confidence,proposed_relative_target
0,00_ASSET_MASTER\01_IDENTITY_REGISTERS\BOC0150-...,review,filename_not_in_canonical_pattern,low,
1,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,move_to_special_folder,duplicate_exact_hash,high,
2,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,move_to_special_folder,duplicate_exact_hash,high,
3,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,move_to_special_folder,duplicate_exact_hash,high,
4,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,move_to_special_folder,duplicate_exact_hash,high,


In [5]:
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'rule_classification_{stamp}'
csv_path, parquet_path = save_rule_outputs(classified, output_base)
print('CSV:', csv_path)
print('Parquet:', parquet_path)


CSV: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\rule_classification_20260330_083147\rule_classification_20260330_083147.csv
Parquet: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\rule_classification_20260330_083147\rule_classification_20260330_083147.parquet


In [6]:
classified.groupby('rule_status').size().sort_values(ascending=False).to_frame('count')

,count
rule_status,
review,192
move_to_special_folder,54
archive_or_delete_candidate,3


In [7]:
classified[classified['rule_status'] == 'archive_or_delete_candidate'][['relative_path', 'filename', 'rule_reason', 'proposed_relative_target']].head(10)

,relative_path,filename,rule_reason,proposed_relative_target
104,03_PERMITTING_APPROVALS\01_PERMITS_LICENSES\04...,.DS_Store,junk_system_or_temp_file,
180,05_COMMERCIAL_PROCUREMENT\02_PURCHASE_ORDERS_S...,.DS_Store,junk_system_or_temp_file,
243,05_COMMERCIAL_PROCUREMENT\99_MISC_REVIEW\ΠΡΟΣΦ...,.DS_Store,junk_system_or_temp_file,


In [8]:
cols = [c for c in ['relative_path', 'filename', 'rule_reason', 'special_folder_target', 'proposed_relative_target'] if c in classified.columns]
classified[classified['rule_status'] == 'move_to_special_folder'][cols].head(10)

,relative_path,filename,rule_reason,special_folder_target,proposed_relative_target
1,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,20260206_Γεν.Πιστοποιητικό_AETHERBIOCHARTECH.pdf,duplicate_exact_hash,_DUPLICATED,
2,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,20260206_Κωδικοποιημένο Καταστατικό_AETHERBIOC...,duplicate_exact_hash,_DUPLICATED,
3,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,20260206_Πιστοποιητικό Αναλ. Εκπρ._AETHERBIOCH...,duplicate_exact_hash,_DUPLICATED,
4,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,20260206_Πιστοποιητικό Ισχ. Εκπρ._AETHERBIOCHA...,duplicate_exact_hash,_DUPLICATED,
24,00_ASSET_MASTER\02_MASTER_DATA\BOC0150-01_DE_S...,BOC0150-01_DE_SDY_carbon_credit_20260130_v01_R...,duplicate_exact_hash,_DUPLICATED,
25,00_ASSET_MASTER\02_MASTER_DATA\BOC0150-01_PM_P...,BOC0150-01_PM_PRS_SCH_short_yyyymmdd_v01_APPRO...,duplicate_exact_hash,_DUPLICATED,
26,00_ASSET_MASTER\02_MASTER_DATA\BOC0150-01_PM_P...,BOC0150-01_PM_PRS_SCH_short_yyyymmdd_v01_APPRO...,duplicate_exact_hash,_DUPLICATED,
29,00_ASSET_MASTER\02_MASTER_DATA\Carbonation Ma...,Carbonation Machine Quotation from Guanma Mac...,duplicate_exact_hash,_DUPLICATED,
31,00_ASSET_MASTER\02_MASTER_DATA\JT- Eva 2500TY ...,JT- Eva 2500TY output agricultural waste carbo...,duplicate_exact_hash,_DUPLICATED,
32,00_ASSET_MASTER\02_MASTER_DATA\PROPOSAL FOR MJ...,PROPOSAL FOR MJT-2000 MODEL BIOCHAR PRODUCTION...,duplicate_exact_hash,_DUPLICATED,


In [9]:
classified[classified['rule_status'] == 'compliant_keep_review_path'][['relative_path', 'filename', 'parsed_phase', 'parsed_doc_type', 'default_folder_subpath', 'proposed_relative_target']].head(10)

,relative_path,filename,parsed_phase,parsed_doc_type,default_folder_subpath,proposed_relative_target


In [10]:
classified[classified['rule_status'] == 'review'][['relative_path', 'filename', 'rule_reason', 'path_risk', 'filename_risk']].head(10)

,relative_path,filename,rule_reason,path_risk,filename_risk
0,00_ASSET_MASTER\01_IDENTITY_REGISTERS\BOC0150-...,BOC0150-01_DE_LEG_company_legal_documents_2026...,filename_not_in_canonical_pattern,,
5,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,1. Biochar - Scaling.pdf,filename_not_in_canonical_pattern,,
6,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,1. Biochar.docx,filename_not_in_canonical_pattern,,
7,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,1. EN-Understanding-Carbon-Dioxide-Removal.pdf,filename_not_in_canonical_pattern,,
8,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,1. European-Biochar-Market-Report_2025.pdf,filename_not_in_canonical_pattern,,
9,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,1. Factsheet_-_Certification_of_carbon_removal...,filename_not_in_canonical_pattern,,
10,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,1. MARKET INSIGHT - Carbon Market 2025.pdf,filename_not_in_canonical_pattern,,
11,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,1. eu goals and biochar.pdf,filename_not_in_canonical_pattern,,
12,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,Biomass residue to carbon dioxide removal - Bi...,filename_not_in_canonical_pattern,,
13,00_ASSET_MASTER\02_MASTER_DATA\01 - Τι είναι τ...,Carbon Removal - Biomass to Biochar - Base 25.pdf,filename_not_in_canonical_pattern,,


In [11]:
classified.sort_values(['action_priority', 'relative_path']).head(10)

,scan_root,absolute_path,relative_path,parent_relative,filename,stem,suffix,size_bytes,modified_at,created_at,...,routing_basis,counterparty_rule_ok,counterparty_rule_reason,default_folder_subpath,current_folder_subpath,path_length_warning,filename_length_warning,path_risk,filename_risk,action_priority
104,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,03_PERMITTING_APPROVALS\01_PERMITS_LICENSES\04...,03_PERMITTING_APPROVALS\01_PERMITS_LICENSES\04...,.DS_Store,.DS_Store,,6148,2026-01-21 18:10:31.275055170,2026-03-30 05:15:42.330575705,...,,False,not_applicable,,03_PERMITTING_APPROVALS/01_PERMITS_LICENSES/04...,False,False,,,10
180,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,05_COMMERCIAL_PROCUREMENT\02_PURCHASE_ORDERS_S...,05_COMMERCIAL_PROCUREMENT\02_PURCHASE_ORDERS_S...,.DS_Store,.DS_Store,,6148,2025-10-27 08:45:28.000000000,2026-03-30 05:14:21.121632099,...,,False,not_applicable,,05_COMMERCIAL_PROCUREMENT/02_PURCHASE_ORDERS_S...,False,False,,,10
243,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,05_COMMERCIAL_PROCUREMENT\99_MISC_REVIEW\ΠΡΟΣΦ...,05_COMMERCIAL_PROCUREMENT\99_MISC_REVIEW\ΠΡΟΣΦ...,.DS_Store,.DS_Store,,6148,2025-11-11 13:39:14.567287922,2026-03-30 05:16:41.929895401,...,,False,not_applicable,,05_COMMERCIAL_PROCUREMENT/99_MISC_REVIEW/ΠΡΟΣΦ...,False,False,,,10
1,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,20260206_Γεν.Πιστοποιητικό_AETHERBIOCHARTECH.pdf,20260206_Γεν.Πιστοποιητικό_AETHERBIOCHARTECH,.pdf,85343,2026-02-09 08:11:57.928837299,2026-03-10 13:36:05.589940786,...,,False,not_applicable,,00_ASSET_MASTER/01_IDENTITY_REGISTERS/ΓΕΜΗ_AET...,False,False,,,20
2,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,20260206_Κωδικοποιημένο Καταστατικό_AETHERBIOC...,20260206_Κωδικοποιημένο Καταστατικό_AETHERBIOC...,.pdf,224444,2026-02-09 08:11:57.964921951,2026-03-10 13:36:05.852858782,...,,False,not_applicable,,00_ASSET_MASTER/01_IDENTITY_REGISTERS/ΓΕΜΗ_AET...,False,False,,,20
3,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,20260206_Πιστοποιητικό Αναλ. Εκπρ._AETHERBIOCH...,20260206_Πιστοποιητικό Αναλ. Εκπρ._AETHERBIOCH...,.pdf,135040,2026-02-09 08:11:57.951070786,2026-03-10 13:36:06.097070694,...,,False,not_applicable,,00_ASSET_MASTER/01_IDENTITY_REGISTERS/ΓΕΜΗ_AET...,False,False,,,20
4,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,00_ASSET_MASTER\01_IDENTITY_REGISTERS\ΓΕΜΗ_AET...,20260206_Πιστοποιητικό Ισχ. Εκπρ._AETHERBIOCHA...,20260206_Πιστοποιητικό Ισχ. Εκπρ._AETHERBIOCHA...,.pdf,134546,2026-02-09 08:11:57.938786030,2026-03-10 13:36:06.326879025,...,,False,not_applicable,,00_ASSET_MASTER/01_IDENTITY_REGISTERS/ΓΕΜΗ_AET...,False,False,,,20
24,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,00_ASSET_MASTER\02_MASTER_DATA\BOC0150-01_DE_S...,00_ASSET_MASTER\02_MASTER_DATA,BOC0150-01_DE_SDY_carbon_credit_20260130_v01_R...,BOC0150-01_DE_SDY_carbon_credit_20260130_v01_R...,.pdf,487334,2026-01-20 12:23:37.192737103,2026-03-10 13:29:09.471312523,...,,False,not_applicable,,00_ASSET_MASTER/02_MASTER_DATA,False,False,,,20
25,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,C:\Users\User\Desktop\ATHBIO-IKE-803026649\ASS...,00_ASSET_MASTER\02_MASTER_DATA\BOC0150-01_PM_P...,00_ASSET_MASTER\02_MASTER_DATA,BOC0150-01_PM_PRS_SCH_short_yyyymmdd_v01_APPRO...,BOC0150-01_PM_